# v7 — Mejoras para Detección de Melanoma

**Diagnóstico v6 (Grad-CAM):**
- Activación central promedio = 0.36 → el modelo mira artefactos (pelo, fondo), no la lesión
- 60% de mel predicho como nv, 32% como bkl, 0% correcto
- El modelo nunca predice mel en test: clase completamente ignorada

**Mejoras implementadas en v7:**
1. **Hair removal** — filtro inpainting para eliminar pelo antes de entrenar
2. **Oversample mel ×5** — balancear el dataset de entrenamiento
3. **Class weights agresivos** — mel=10, nv=0.3, resto proporcional
4. **Augmentation específica** para mel — rotaciones, zoom, color jitter
5. Misma arquitectura DenseNet201 + Focal Loss (gamma=2.0)


## 1. Verificar entorno

In [1]:
import os, sys, gc
import numpy as np
import tensorflow as tf
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import cv2

print('TensorFlow:', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))
print('NumPy:', np.__version__)

# Semilla de reproducibilidad
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)


TensorFlow: 2.21.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
NumPy: 2.1.3


## 2. Configuración

In [2]:
NUMPY_DIR  = '/kaggle/input/datasets/michellercarranza/ham10000-numpy'
OUTPUT_DIR = '/kaggle/working/v7'
os.makedirs(OUTPUT_DIR, exist_ok=True)

CLASS_NAMES = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
N_CLASSES   = len(CLASS_NAMES)
MEL_IDX     = CLASS_NAMES.index('mel')  # 4
NV_IDX      = CLASS_NAMES.index('nv')   # 5
IMG_SIZE    = 224
BATCH_SIZE  = 32

# Hiperparámetros v7
MEL_OVERSAMPLE_FACTOR = 5    # repetir muestras de mel 5 veces
MEL_CLASS_WEIGHT      = 10.0 # peso para mel en la loss
NV_CLASS_WEIGHT       = 0.3  # peso para nv (clase dominante)

print('Configuración lista.')
print(f'  MEL_OVERSAMPLE_FACTOR = {MEL_OVERSAMPLE_FACTOR}')
print(f'  MEL_CLASS_WEIGHT      = {MEL_CLASS_WEIGHT}')
print(f'  NV_CLASS_WEIGHT       = {NV_CLASS_WEIGHT}')


Configuración lista.
  MEL_OVERSAMPLE_FACTOR = 5
  MEL_CLASS_WEIGHT      = 10.0
  NV_CLASS_WEIGHT       = 0.3


## 3. Cargar datos

In [3]:
print('Cargando datos...')
X_train = np.load(os.path.join(NUMPY_DIR, 'train_X.npy'))
y_train = np.load(os.path.join(NUMPY_DIR, 'train_y.npy'))
X_val   = np.load(os.path.join(NUMPY_DIR, 'validation_X.npy'))
y_val   = np.load(os.path.join(NUMPY_DIR, 'validation_y.npy'))
X_test  = np.load(os.path.join(NUMPY_DIR, 'test_X.npy'))
y_test  = np.load(os.path.join(NUMPY_DIR, 'test_y.npy'))

print(f'Train: {X_train.shape}  y: {y_train.shape}  dtype: {y_train.dtype}')
print(f'Val:   {X_val.shape}')
print(f'Test:  {X_test.shape}')

# y es int32 (índice de clase)
y_train_int = y_train
y_val_int   = y_val
y_test_int  = y_test

# Distribución original
print('\nDistribución original en train:')
for i, name in enumerate(CLASS_NAMES):
    n = np.sum(y_train_int == i)
    print(f'  {name:6s}: {n:5d} ({n/len(y_train_int)*100:.1f}%)')


Cargando datos...
Train: (9577, 224, 224, 3)  y: (9577,)  dtype: int32
Val:   (2492, 224, 224, 3)
Test:  (1285, 224, 224, 3)

Distribución original en train:
  akiec :   315 (3.3%)
  bcc   :   487 (5.1%)
  bkl   :  1048 (10.9%)
  df    :   110 (1.1%)
  mel   :  1076 (11.2%)
  nv    :  6405 (66.9%)
  vasc  :   136 (1.4%)


## 4. Hair Removal (preprocesamiento)

Eliminar el pelo de las imágenes dermoscópicas usando:
1. **Blackhat morphology** — detecta estructuras oscuras sobre fondo claro (pelos)
2. **Inpainting** — rellena las zonas detectadas con contexto circundante


In [4]:
def remove_hair(img_uint8, kernel_size=17, threshold=10):
    """
    Elimina el pelo de una imagen dermoscópica.
    img_uint8: (H, W, 3) uint8 [0-255]
    Returns: imagen sin pelo, máscara del pelo
    """
    # Convertir a grayscale
    gray = cv2.cvtColor(img_uint8, cv2.COLOR_RGB2GRAY)

    # Blackhat: detecta estructuras oscuras (pelos) sobre fondo claro
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (kernel_size, kernel_size))
    blackhat = cv2.morphologyEx(gray, cv2.MORPH_BLACKHAT, kernel)

    # Umbralizar para obtener máscara binaria del pelo
    _, mask = cv2.threshold(blackhat, threshold, 255, cv2.THRESH_BINARY)

    # Dilatar máscara para cubrir completamente el pelo
    kernel_dilate = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    mask = cv2.dilate(mask, kernel_dilate, iterations=1)

    # Inpainting: rellenar zonas enmascaradas
    # Convertir a BGR para cv2.inpaint
    img_bgr = cv2.cvtColor(img_uint8, cv2.COLOR_RGB2BGR)
    inpainted_bgr = cv2.inpaint(img_bgr, mask, inpaintRadius=3, flags=cv2.INPAINT_TELEA)
    inpainted_rgb = cv2.cvtColor(inpainted_bgr, cv2.COLOR_BGR2RGB)

    return inpainted_rgb, mask


def preprocess_batch(X_float32, kernel_size=17, threshold=10):
    """
    Aplica hair removal a un batch de imágenes float32 [0,1].
    Returns: batch float32 [0,1] sin pelo
    """
    result = np.empty_like(X_float32)
    for i in range(len(X_float32)):
        # Desnormalizar a uint8
        # Las imágenes están normalizadas con ImageNet mean/std
        # Revertir: x_orig = x * std + mean, luego *255
        mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
        std  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
        img_denorm = X_float32[i] * std + mean
        img_uint8 = np.clip(img_denorm * 255, 0, 255).astype(np.uint8)

        # Aplicar hair removal
        img_clean, _ = remove_hair(img_uint8, kernel_size, threshold)

        # Volver a normalizar
        img_renorm = (img_clean.astype(np.float32) / 255.0 - mean) / std
        result[i] = img_renorm
    return result


print('Función hair removal definida.')
print('Probando en 3 imágenes...')

# Probar visualmente con 3 imágenes de mel
mel_indices = np.where(y_train_int == MEL_IDX)[0]
sample_indices = mel_indices[:3]

mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
std  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

fig, axes = plt.subplots(3, 3, figsize=(10, 10))
for row, idx in enumerate(sample_indices):
    img_f = X_train[idx]
    img_uint8 = np.clip((img_f * std + mean) * 255, 0, 255).astype(np.uint8)
    img_clean, mask = remove_hair(img_uint8)

    axes[row][0].imshow(img_uint8)
    axes[row][0].set_title('Original', fontsize=9)
    axes[row][0].axis('off')
    axes[row][1].imshow(mask, cmap='gray')
    axes[row][1].set_title('Máscara pelo', fontsize=9)
    axes[row][1].axis('off')
    axes[row][2].imshow(img_clean)
    axes[row][2].set_title('Sin pelo', fontsize=9)
    axes[row][2].axis('off')

plt.suptitle('Hair Removal — Muestras de Melanoma', fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'hair_removal_test.png'), dpi=120, bbox_inches='tight')
plt.show()
print('Guardada: hair_removal_test.png')


Función hair removal definida.
Probando en 3 imágenes...
Guardada: hair_removal_test.png


## 5. Aplicar Hair Removal a todo el dataset

In [5]:
print('Aplicando hair removal a train...')
CHUNK = 500
X_train_clean = np.empty_like(X_train)
for i in range(0, len(X_train), CHUNK):
    end = min(i + CHUNK, len(X_train))
    X_train_clean[i:end] = preprocess_batch(X_train[i:end])
    if (i // CHUNK + 1) % 5 == 0:
        print(f'  {end}/{len(X_train)} procesadas')
print('Train listo.')

print('Aplicando hair removal a val...')
X_val_clean = preprocess_batch(X_val)
print('Val listo.')

print('Aplicando hair removal a test...')
X_test_clean = preprocess_batch(X_test)
print('Test listo.')

# Liberar memoria
del X_train, X_val, X_test
gc.collect()
print('Memoria liberada.')


Aplicando hair removal a train...
  2500/9577 procesadas
  5000/9577 procesadas
  7500/9577 procesadas
  9577/9577 procesadas
Train listo.
Aplicando hair removal a val...
Val listo.
Aplicando hair removal a test...
Test listo.
Memoria liberada.


## 6. Oversample de Melanoma

Repetir las muestras de mel en train para balancear el dataset.
Factor ×5 → mel pasa de ~1115 a ~5575 muestras.


In [6]:
mel_train_indices = np.where(y_train_int == MEL_IDX)[0]
print(f'Muestras de mel en train original: {len(mel_train_indices)}')

# Repetir mel MEL_OVERSAMPLE_FACTOR veces
X_mel_extra = np.tile(X_train_clean[mel_train_indices], (MEL_OVERSAMPLE_FACTOR - 1, 1, 1, 1))
y_mel_extra = np.tile(y_train_int[mel_train_indices], MEL_OVERSAMPLE_FACTOR - 1)

X_train_aug = np.concatenate([X_train_clean, X_mel_extra], axis=0)
y_train_aug = np.concatenate([y_train_int, y_mel_extra], axis=0)

# Mezclar
perm = np.random.permutation(len(X_train_aug))
X_train_aug = X_train_aug[perm]
y_train_aug = y_train_aug[perm]

print(f'Train aumentado: {X_train_aug.shape}')
print('\nDistribución train aumentado:')
for i, name in enumerate(CLASS_NAMES):
    n = np.sum(y_train_aug == i)
    print(f'  {name:6s}: {n:5d} ({n/len(y_train_aug)*100:.1f}%)')


Muestras de mel en train original: 1076


## 7. Class Weights

In [ ]:
# Calcular class weights base (inverso de frecuencia)
counts = np.array([np.sum(y_train_aug == i) for i in range(N_CLASSES)], dtype=np.float32)
base_weights = counts.max() / counts

# Ajuste manual agresivo
class_weights = {}
for i, name in enumerate(CLASS_NAMES):
    if i == MEL_IDX:
        w = MEL_CLASS_WEIGHT
    elif i == NV_IDX:
        w = NV_CLASS_WEIGHT
    else:
        w = float(base_weights[i])
    class_weights[i] = w
    print(f'  {name:6s}: weight = {w:.3f}')

print('\nClass weights configurados.')


## 8. Data Augmentation con augmentación agresiva para mel

Augmentation más fuerte para mel vs otras clases.


In [ ]:
def augment_image(img, is_mel=False):
    """
    Augmentation para una imagen.
    is_mel: True → augmentation más agresiva
    """
    img = img.copy()

    # Flip horizontal/vertical
    if np.random.rand() > 0.5:
        img = np.fliplr(img)
    if np.random.rand() > 0.5:
        img = np.flipud(img)

    # Rotación (mel: hasta 360°, otros: hasta 90°)
    max_angle = 360 if is_mel else 90
    angle = np.random.uniform(-max_angle, max_angle)
    h, w = img.shape[:2]
    M = cv2.getRotationMatrix2D((w/2, h/2), angle, 1.0)
    # Reconstruir uint8 temporalmente para cv2
    mean_v = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std_v  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
    img_uint8 = np.clip((img * std_v + mean_v) * 255, 0, 255).astype(np.uint8)
    img_uint8 = cv2.warpAffine(img_uint8, M, (w, h),
                                borderMode=cv2.BORDER_REFLECT)
    img = (img_uint8.astype(np.float32) / 255.0 - mean_v) / std_v

    if is_mel:
        # Zoom aleatorio (0.85 – 1.15)
        scale = np.random.uniform(0.85, 1.15)
        new_h, new_w = int(h * scale), int(w * scale)
        img_uint8_2 = np.clip((img * std_v + mean_v) * 255, 0, 255).astype(np.uint8)
        resized = cv2.resize(img_uint8_2, (new_w, new_h))
        if scale > 1.0:  # crop al centro
            dh = (new_h - h) // 2
            dw = (new_w - w) // 2
            resized = resized[dh:dh+h, dw:dw+w]
        else:  # pad
            padded = np.zeros((h, w, 3), dtype=np.uint8)
            dh = (h - new_h) // 2
            dw = (w - new_w) // 2
            padded[dh:dh+new_h, dw:dw+new_w] = resized
            resized = padded
        img = (resized.astype(np.float32) / 255.0 - mean_v) / std_v

        # Color jitter (brillo y contraste leve)
        alpha = np.random.uniform(0.85, 1.15)  # contraste
        beta  = np.random.uniform(-0.05, 0.05)  # brillo (en espacio normalizado)
        img = img * alpha + beta

    return img.astype(np.float32)


print('Función augment_image definida.')
print('Probando augmentation en una imagen de mel...')

# Vista previa de augmentation
sample_mel = X_train_clean[mel_train_indices[0]]
mean_v = np.array([0.485, 0.456, 0.406], dtype=np.float32)
std_v  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
axes[0][0].imshow(np.clip(sample_mel * std_v + mean_v, 0, 1))
axes[0][0].set_title('Original', fontsize=8)
axes[0][0].axis('off')
for j in range(1, 5):
    aug = augment_image(sample_mel, is_mel=True)
    axes[0][j].imshow(np.clip(aug * std_v + mean_v, 0, 1))
    axes[0][j].set_title(f'Aug mel {j}', fontsize=8)
    axes[0][j].axis('off')

nv_indices_train = np.where(y_train_int == NV_IDX)[0]
sample_nv = X_train_clean[nv_indices_train[0]]
axes[1][0].imshow(np.clip(sample_nv * std_v + mean_v, 0, 1))
axes[1][0].set_title('nv Original', fontsize=8)
axes[1][0].axis('off')
for j in range(1, 5):
    aug = augment_image(sample_nv, is_mel=False)
    axes[1][j].imshow(np.clip(aug * std_v + mean_v, 0, 1))
    axes[1][j].set_title(f'Aug nv {j}', fontsize=8)
    axes[1][j].axis('off')

plt.suptitle('Augmentation: mel (arriba) vs nv (abajo)', fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'augmentation_preview.png'), dpi=120, bbox_inches='tight')
plt.show()
print('Guardada: augmentation_preview.png')


## 9. Generador de datos con augmentation

In [ ]:
class AugmentedGenerator(tf.keras.utils.Sequence):
    """Generador con augmentation diferenciada por clase."""
    def __init__(self, X, y, batch_size, mel_idx, augment=True, shuffle=True):
        self.X = X
        self.y = y
        self.batch_size = batch_size
        self.mel_idx = mel_idx
        self.augment = augment
        self.shuffle = shuffle
        self.indices = np.arange(len(X))
        if shuffle:
            np.random.shuffle(self.indices)

    def __len__(self):
        return int(np.ceil(len(self.X) / self.batch_size))

    def __getitem__(self, idx):
        batch_idx = self.indices[idx * self.batch_size:(idx + 1) * self.batch_size]
        X_batch = np.empty((len(batch_idx), IMG_SIZE, IMG_SIZE, 3), dtype=np.float32)
        y_batch = self.y[batch_idx]

        for i, j in enumerate(batch_idx):
            img = self.X[j]
            if self.augment:
                is_mel = (self.y[j] == self.mel_idx)
                img = augment_image(img, is_mel=is_mel)
            X_batch[i] = img

        # One-hot encoding
        y_onehot = tf.keras.utils.to_categorical(y_batch, num_classes=N_CLASSES)
        return X_batch, y_onehot

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)


train_gen = AugmentedGenerator(X_train_aug, y_train_aug, BATCH_SIZE, MEL_IDX, augment=True)
val_gen   = AugmentedGenerator(X_val_clean, y_val_int, BATCH_SIZE, MEL_IDX, augment=False, shuffle=False)

print(f'Train batches: {len(train_gen)}')
print(f'Val batches:   {len(val_gen)}')

# Verificar un batch
X_b, y_b = train_gen[0]
print(f'Batch shape: X={X_b.shape}  y={y_b.shape}')
print(f'Clases en batch: {np.argmax(y_b, axis=1)[:10]}')


## 10. Construcción del modelo DenseNet201

In [ ]:
def build_model(n_classes=N_CLASSES, img_size=IMG_SIZE):
    base = tf.keras.applications.DenseNet201(
        include_top=False,
        weights='imagenet',
        input_shape=(img_size, img_size, 3)
    )
    base.trainable = False  # Fase 1: congelar todo

    inputs = tf.keras.Input(shape=(img_size, img_size, 3))
    x = base(inputs, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dense(256, activation='relu',
                               kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
    x = tf.keras.layers.Dropout(0.5)(x)
    outputs = tf.keras.layers.Dense(n_classes, activation='softmax')(x)

    model = tf.keras.Model(inputs, outputs)
    return model, base


class FocalLoss(tf.keras.losses.Loss):
    def __init__(self, gamma=2.0, alpha=None, **kwargs):
        super().__init__(**kwargs)
        self.gamma = gamma
        self.alpha = alpha

    def call(self, y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        ce = -y_true * tf.math.log(y_pred)
        pt = tf.reduce_sum(y_true * y_pred, axis=-1, keepdims=True)
        focal_weight = tf.pow(1.0 - pt, self.gamma)
        loss = focal_weight * ce
        return tf.reduce_mean(tf.reduce_sum(loss, axis=-1))

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'gamma': self.gamma, 'alpha': self.alpha})
        return cfg


model, base_model = build_model()
model.summary()
print(f'Parámetros totales:     {model.count_params():,}')
print(f'Parámetros entrenables: {sum(p.numpy().size for p in model.trainable_weights):,}')


## 11. Fase 1 — Entrenar cabeza (base congelada)

10 épocas, lr=1e-3, CrossEntropy (no Focal aún).


In [ ]:
PHASE1_EPOCHS = 10
PHASE1_LR     = 1e-3

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=PHASE1_LR),
    loss=tf.keras.losses.CategoricalCrossentropy(),
    metrics=['accuracy']
)

cb_p1 = [
    tf.keras.callbacks.ModelCheckpoint(
        os.path.join(OUTPUT_DIR, 'densenet201_v7_phase1_best.keras'),
        monitor='val_accuracy', save_best_only=True, verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1
    )
]

print(f'Entrenando Fase 1 ({PHASE1_EPOCHS} épocas, lr={PHASE1_LR})...')
history_p1 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=PHASE1_EPOCHS,
    callbacks=cb_p1,
    class_weight=class_weights,
    verbose=1
)
print('Fase 1 completada.')


## 12. Fase 2a — Descongelar últimas 25 capas + Focal Loss

In [ ]:
PHASE2A_UNFREEZE = 25
PHASE2A_EPOCHS   = 15
PHASE2A_LR       = 2e-4

# Descongelar últimas N capas del base
base_model.trainable = True
for layer in base_model.layers[:-PHASE2A_UNFREEZE]:
    layer.trainable = False

trainable_now = sum(1 for l in base_model.layers if l.trainable)
print(f'Capas entrenables en base: {trainable_now} / {len(base_model.layers)}')

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=PHASE2A_LR),
    loss=FocalLoss(gamma=2.0),
    metrics=['accuracy']
)

cb_p2a = [
    tf.keras.callbacks.ModelCheckpoint(
        os.path.join(OUTPUT_DIR, 'densenet201_v7_phase2a_best.keras'),
        monitor='val_accuracy', save_best_only=True, verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1
    )
]

print(f'Entrenando Fase 2a ({PHASE2A_EPOCHS} épocas, lr={PHASE2A_LR}, Focal Loss)...')
history_p2a = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=PHASE2A_EPOCHS,
    callbacks=cb_p2a,
    class_weight=class_weights,
    verbose=1
)
print('Fase 2a completada.')


## 13. Fase 2b — Descongelar últimas 100 capas

In [ ]:
PHASE2B_UNFREEZE = 100
PHASE2B_EPOCHS   = 20
PHASE2B_LR       = 5e-5

for layer in base_model.layers[:-PHASE2B_UNFREEZE]:
    layer.trainable = False
for layer in base_model.layers[-PHASE2B_UNFREEZE:]:
    layer.trainable = True

trainable_now = sum(1 for l in base_model.layers if l.trainable)
print(f'Capas entrenables en base: {trainable_now} / {len(base_model.layers)}')

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=PHASE2B_LR),
    loss=FocalLoss(gamma=2.0),
    metrics=['accuracy']
)

cb_p2b = [
    tf.keras.callbacks.ModelCheckpoint(
        os.path.join(OUTPUT_DIR, 'densenet201_v7_phase2b_best.keras'),
        monitor='val_accuracy', save_best_only=True, verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=7, restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1
    )
]

print(f'Entrenando Fase 2b ({PHASE2B_EPOCHS} épocas, lr={PHASE2B_LR}, Focal Loss)...')
history_p2b = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=PHASE2B_EPOCHS,
    callbacks=cb_p2b,
    class_weight=class_weights,
    verbose=1
)
print('Fase 2b completada.')


## 14. Evaluación final en test set

In [ ]:
print('Evaluando en test set...')
y_pred_probs = model.predict(X_test_clean, batch_size=64, verbose=1)
y_pred_int   = np.argmax(y_pred_probs, axis=1)

print('\nReporte de clasificación — v7:')
print(classification_report(y_test_int, y_pred_int, target_names=CLASS_NAMES, zero_division=0))

# Comparar con v5
from sklearn.metrics import f1_score, accuracy_score
acc_v7 = accuracy_score(y_test_int, y_pred_int)
f1_v7  = f1_score(y_test_int, y_pred_int, average='macro', zero_division=0)
f1_mel = f1_score(y_test_int == MEL_IDX, y_pred_int == MEL_IDX, zero_division=0)

print(f'\n--- RESULTADOS v7 ---')
print(f'Accuracy:       {acc_v7:.4f}  (v5: 0.6879)')
print(f'F1 Macro:       {f1_v7:.4f}   (v5: 0.4362)')
print(f'F1 Melanoma:    {f1_mel:.4f}   (v5: 0.0000)')


In [ ]:
# Matriz de confusión v7
cm = confusion_matrix(y_test_int, y_pred_int)

plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='OrRd',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            linewidths=0.5)
plt.title('Matriz de confusión — v7 Test Set', fontsize=14, pad=12)
plt.ylabel('Clase real', fontsize=11)
plt.xlabel('Clase predicha', fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'confusion_matrix_v7.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Guardada: confusion_matrix_v7.png')


## 15. Curvas de entrenamiento

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Concatenar historial de las 3 fases
all_acc     = history_p1.history['accuracy'] + history_p2a.history['accuracy'] + history_p2b.history['accuracy']
all_val_acc = history_p1.history['val_accuracy'] + history_p2a.history['val_accuracy'] + history_p2b.history['val_accuracy']
all_loss    = history_p1.history['loss'] + history_p2a.history['loss'] + history_p2b.history['loss']
all_val_loss= history_p1.history['val_loss'] + history_p2a.history['val_loss'] + history_p2b.history['val_loss']

epochs_total = len(all_acc)
ep1_end = len(history_p1.history['accuracy'])
ep2a_end = ep1_end + len(history_p2a.history['accuracy'])

axes[0].plot(all_acc, label='Train', color='#1a6e35')
axes[0].plot(all_val_acc, label='Val', color='#c85a08')
axes[0].axvline(ep1_end, color='gray', linestyle='--', alpha=0.7, label='Fase 2a')
axes[0].axvline(ep2a_end, color='black', linestyle='--', alpha=0.7, label='Fase 2b')
axes[0].set_title('Accuracy')
axes[0].legend()

axes[1].plot(all_loss, label='Train', color='#1a6e35')
axes[1].plot(all_val_loss, label='Val', color='#c85a08')
axes[1].axvline(ep1_end, color='gray', linestyle='--', alpha=0.7)
axes[1].axvline(ep2a_end, color='black', linestyle='--', alpha=0.7)
axes[1].set_title('Loss')
axes[1].legend()

plt.suptitle('Curvas de entrenamiento — v7', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'training_curves_v7.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Guardada: training_curves_v7.png')


## 16. Resumen de archivos generados

In [ ]:
print('=' * 60)
print('RESUMEN v7')
print('=' * 60)
print(f'Accuracy:    {acc_v7:.4f}')
print(f'F1 Macro:    {f1_v7:.4f}')
print(f'F1 Melanoma: {f1_mel:.4f}')
print()
print('Archivos en', OUTPUT_DIR)
for f in sorted(os.listdir(OUTPUT_DIR)):
    fp = os.path.join(OUTPUT_DIR, f)
    print(f'  {f:50s}  {os.path.getsize(fp)/1e6:.1f} MB')
